In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("./wildfires_dataset.sqlite")

query = """
SELECT
    FIRE_YEAR,
    DISCOVERY_DOY,
    STAT_CAUSE_DESCR,
    FIRE_SIZE,
    FIRE_SIZE_CLASS,
    LATITUDE,
    LONGITUDE,
    STATE
FROM Fires
"""

df = pd.read_sql_query(query, conn)

conn.close()

df.info()

In [ ]:
import matplotlib.pyplot as plt

fire_class_order = list("ABCDEFG")

# A = menor incendio, G = maior incendio. O valor controla o tamanho do ponto no scatter.
size_by_class = {
    "A": 3,
    "B": 6,
    "C": 12,
    "D": 22,
    "E": 36,
    "F": 56,
    "G": 82,
}

color_by_class = {
    "A": "#2c7fb8",
    "B": "#41b6c4",
    "C": "#a1dab4",
    "D": "#ffffcc",
    "E": "#fecc5c",
    "F": "#fd8d3c",
    "G": "#e31a1c",
}

plot_df = df.dropna(subset=["LATITUDE", "LONGITUDE", "FIRE_SIZE_CLASS"]).copy()
plot_df = plot_df[plot_df["FIRE_SIZE_CLASS"].isin(fire_class_order)]

# Mantem o mapa focado nos EUA continentais. Remova este filtro para incluir AK/HI/territorios.
plot_df = plot_df[
    plot_df["LATITUDE"].between(24, 50)
    & plot_df["LONGITUDE"].between(-125, -66)
]

fig, ax = plt.subplots(figsize=(14, 8))

for fire_class in fire_class_order:
    class_df = plot_df[plot_df["FIRE_SIZE_CLASS"] == fire_class]

    ax.scatter(
        class_df["LONGITUDE"],
        class_df["LATITUDE"],
        s=size_by_class[fire_class],
        color=color_by_class[fire_class],
        alpha=0.35,
        edgecolors="none",
        label=f"Classe {fire_class}",
    )

ax.set_title("Localizacao dos incendios nos EUA por classe de tamanho")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.35)
ax.legend(title="FIRE_SIZE_CLASS", markerscale=2, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Seleciona apenas as colunas que o modelo vai usar.
# Entrada do modelo: LATITUDE e LONGITUDE.
# Saida que queremos prever: FIRE_SIZE_CLASS.
knn_df = df[["LATITUDE", "LONGITUDE", "FIRE_SIZE_CLASS"]].dropna().copy()
knn_df = knn_df[knn_df["FIRE_SIZE_CLASS"].isin(list("ABCDEFG"))]

TRAIN_SIZE = 1_000_000
TEST_SIZE = 20_000
K_NEIGHBORS = 30

# 2. Sorteia os registros, porque a base esta ordenada e queremos uma amostra aleatoria.
# Depois separamos 1 milhao para treino e 20 mil para teste.
knn_sample = knn_df.sample(n=TRAIN_SIZE + TEST_SIZE, random_state=42)

X = knn_sample[["LATITUDE", "LONGITUDE"]]
y = knn_sample["FIRE_SIZE_CLASS"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=y,
)

# 3. Cria e treina o modelo KNN.
knn_model = KNeighborsClassifier(n_neighbors=K_NEIGHBORS, n_jobs=-1)
knn_model.fit(X_train, y_train)

# 4. Testa o modelo com os 20 mil registros separados para teste.
y_pred = knn_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Acuracia do KNN com k={K_NEIGHBORS}: {accuracy:.2%}")

# 5. Funcao para prever a classe a partir de uma nova latitude e longitude.
def prever_fire_size_class(latitude, longitude):
    entrada = pd.DataFrame(
        {
            "LATITUDE": [latitude],
            "LONGITUDE": [longitude],
        }
    )

    return knn_model.predict(entrada)[0]

# Exemplo usando uma coordenada da propria base.
latitude_entrada = 40.968056
longitude_entrada = -122.433889
classe_prevista = prever_fire_size_class(latitude_entrada, longitude_entrada)

print(
    f"Classe prevista para latitude={latitude_entrada} "
    f"e longitude={longitude_entrada}: {classe_prevista}"
)


In [91]:
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "FIRE_SIZE_CLASS"
BLOCKED_FEATURES = {"FIRE_SIZE", "FIRE_SIZE_CLASS"}
TRAIN_SIZE = 1_000_000
TEST_SIZE = 20_000
K_NEIGHBORS = 35


# Cada item aqui sera testado como um conjunto diferente de entrada para o KNN.
# FIRE_SIZE fica de fora porque tem relacao direta com FIRE_SIZE_CLASS e vazaria a resposta.
FEATURES = {
    "localizacao": ["LATITUDE", "LONGITUDE"],
    "localizacao_estado_causa": ["LATITUDE", "LONGITUDE", "STATE", "DISCOVERY_DOY", "STAT_CAUSE_DESCR"],
    "tempo": ["FIRE_YEAR", "DISCOVERY_DOY"],
    "causa": ["STAT_CAUSE_DESCR"],
    "estado": ["STATE"],
    "localizacao_tempo": ["LATITUDE", "LONGITUDE", "FIRE_YEAR", "DISCOVERY_DOY"],
    "localizacao_causa": ["LATITUDE", "LONGITUDE", "STAT_CAUSE_DESCR"],
    "localizacao_estado": ["LATITUDE", "LONGITUDE", "STATE"],
    "causa_estado": ["STAT_CAUSE_DESCR", "STATE"],
    "todos_sem_fire_size": [ "FIRE_YEAR", "DISCOVERY_DOY", "STAT_CAUSE_DESCR", "LATITUDE", "LONGITUDE", "STATE" ],
}

# Confere se nenhum grupo usa colunas proibidas ou inexistentes.
for feature_name, feature_columns in FEATURES.items():
    blocked_columns = BLOCKED_FEATURES.intersection(feature_columns)
    if blocked_columns:
        raise ValueError(f"{feature_name} usa colunas proibidas: {blocked_columns}")

    missing_columns = set(feature_columns + [TARGET]) - set(df.columns)
    if missing_columns:
        raise ValueError(f"{feature_name} usa colunas que nao existem no df: {missing_columns}")

# Cria uma amostra aleatoria unica para todos os testes, assim a comparacao fica justa.
all_feature_columns = sorted(set().union(*FEATURES.values()))
knn_feature_df = df[[TARGET, *all_feature_columns]].dropna().copy()
knn_feature_df = knn_feature_df[knn_feature_df[TARGET].isin(list("ABCDEFG"))]
knn_sample = knn_feature_df.sample(n=TRAIN_SIZE + TEST_SIZE, random_state=42)

# Separa 1 milhao de linhas para treino e 20 mil para teste.
# stratify mantem a proporcao das classes A-G nos dois conjuntos.
train_df, test_df = train_test_split(
    knn_sample,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    random_state=42,
    stratify=knn_sample[TARGET],
)

y_train = train_df[TARGET]
y_test = test_df[TARGET]


def create_knn_pipeline(feature_columns):
    # KNN depende de distancia, entao as colunas numericas precisam ficar na mesma escala.
    numeric_columns = train_df[feature_columns].select_dtypes(include="number").columns.tolist()
    categorical_columns = [col for col in feature_columns if col not in numeric_columns]

    transformers = []
    if numeric_columns:
        transformers.append(("num", StandardScaler(), numeric_columns))
    if categorical_columns:
        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns))

    return Pipeline(
        steps=[
            ("preprocess", ColumnTransformer(transformers=transformers)),
            ("knn", KNeighborsClassifier(n_neighbors=K_NEIGHBORS, n_jobs=-1)),
        ]
    )


results = []
trained_models = {}

for feature_name, feature_columns in FEATURES.items():
    model = create_knn_pipeline(feature_columns)
    model.fit(train_df[feature_columns], y_train)

    y_pred = model.predict(test_df[feature_columns])
    accuracy = accuracy_score(y_test, y_pred)

    results.append(
        {
            "feature_set": feature_name,
            "accuracy": accuracy,
            "features": feature_columns,
        }
    )
    trained_models[feature_name] = model

    print(f"{feature_name}: {accuracy:.2%}")

results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
display(results_df)

best_feature_name = results_df.iloc[0]["feature_set"]
best_feature_columns = FEATURES[best_feature_name]
best_knn_model = trained_models[best_feature_name]

print(f"Melhor conjunto de features: {best_feature_name}")
print(f"Acuracia: {results_df.iloc[0]['accuracy']:.2%}")
print(f"Colunas usadas: {best_feature_columns}")


localizacao_estado_causa: 62.09%
tempo: 53.25%
causa: 54.68%
estado: 59.44%
localizacao_tempo: 62.24%
localizacao_causa: 63.04%
localizacao_estado: 62.81%
causa_estado: 58.99%
todos_sem_fire_size: 62.69%


,feature_set,accuracy,features
6,localizacao_causa,0.63040,"[LATITUDE, LONGITUDE, STAT_CAUSE_DESCR]"
0,localizacao,0.62815,"[LATITUDE, LONGITUDE]"
7,localizacao_estado,0.62810,"[LATITUDE, LONGITUDE, STATE]"
9,todos_sem_fire_size,0.62685,"[FIRE_YEAR, DISCOVERY_DOY, STAT_CAUSE_DESCR, L..."
5,localizacao_tempo,0.62245,"[LATITUDE, LONGITUDE, FIRE_YEAR, DISCOVERY_DOY]"
1,localizacao_estado_causa,0.62095,"[LATITUDE, LONGITUDE, STATE, DISCOVERY_DOY, ST..."
4,estado,0.59435,[STATE]
8,causa_estado,0.58990,"[STAT_CAUSE_DESCR, STATE]"
3,causa,0.54680,[STAT_CAUSE_DESCR]
2,tempo,0.53250,"[FIRE_YEAR, DISCOVERY_DOY]"


Melhor conjunto de features: localizacao_causa
Acuracia: 63.04%
Colunas usadas: ['LATITUDE', 'LONGITUDE', 'STAT_CAUSE_DESCR']
